In [ ]:
# ==============================================================================
# 📦 1. IMPORTAÇÃO DE BIBLIOTECAS
# ==============================================================================
import os
from dotenv import load_dotenv

import pandas as pd
import psycopg2 as pg
import sqlalchemy
from sqlalchemy import create_engine
import panel as pn

In [ ]:
# Inicializa as extensões do Panel para tabelas e notificações
pn.extension()
pn.extension('tabulator')
pn.extension(notifications=True)

In [ ]:
# ==============================================================================
# 🔐 2. CONFIGURAÇÃO DE VARIÁVEIS DE AMBIENTE
# ==============================================================================
# Carrega as variáveis do arquivo .env
load_dotenv()

# Lê as credenciais para o banco de dados
DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')

In [ ]:
# ==============================================================================
# 🗄️ 3. CONEXÃO COM O BANCO DE DADOS (POSTGRESQL)
# ==============================================================================
# Conexão nativa usando Psycopg2 (usada para cursores, INSERTS, UPDATES e DELETES)
con = pg.connect(
    host=DB_HOST, 
    dbname=DB_NAME, 
    user=DB_USER, 
    password=DB_PASS
)

# String e Engine do SQLAlchemy (usada principalmente pelo Pandas para SELECTS)
cnx = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}'
engine = create_engine(cnx)

In [ ]:
# ==============================================================================
# MÓDULO DE ACESSO
# ==============================================================================
# Variável para ignorar campos vazios na consulta
flag = ''

import datetime
import pandas as pd
import panel as pn

# Widgets de Identificação
id_usr = pn.widgets.IntInput(name="ID do Usuário (Para Alterar/Excluir)", value=0, start=0)
cpf_usr = pn.widgets.TextInput(name="CPF (Apenas números)", value='', max_length=11)
nome_usr = pn.widgets.TextInput(name="Nome Completo", value='')
email_usr = pn.widgets.TextInput(name="E-mail", value='')
senha_usr = pn.widgets.PasswordInput(name="Senha de Acesso", value='')
nascimento_usr = pn.widgets.DatePicker(name="Data de Nascimento", value=datetime.date(1990, 1, 1))

# Widgets de Endereço
cep_usr = pn.widgets.IntInput(name="CEP", value=0)
num_usr = pn.widgets.IntInput(name="Nº", value=0)
bairro_usr = pn.widgets.TextInput(name="Bairro", value='')
cidade_usr = pn.widgets.TextInput(name="Cidade", value='')
estado_usr = pn.widgets.TextInput(name="Estado (UF)", value='', max_length=2)

# Botões de Ação
btn_consultar_usr = pn.widgets.Button(name='Consultar Usuários', button_type='primary')
btn_inserir_usr = pn.widgets.Button(name='Inserir Novo', button_type='success')
btn_atualizar_usr = pn.widgets.Button(name='Atualizar Dados', button_type='warning')
btn_excluir_usr = pn.widgets.Button(name='Excluir Usuário', button_type='danger')

def queryAllUsuarios():
    """Consulta os utilizadores cadastrados. Oculta a senha por segurança."""
    query = """
        SELECT id_usuario, cpf, nome_completo, email, data_nascimento, cidade, estado 
        FROM usuario 
        ORDER BY id_usuario;
    """
    df = pd.read_sql_query(query, cnx)
    
    # Formata a data para um padrão de leitura mais amigável
    if not df.empty and 'data_nascimento' in df.columns:
        df['data_nascimento'] = pd.to_datetime(df['data_nascimento']).dt.strftime('%d/%m/%Y')
        
    return pn.widgets.Tabulator(df, layout='fit_columns', pagination='remote', page_size=10)

def on_inserir_usuario():
    try:
        cursor = con.cursor()
        query = """
            INSERT INTO usuario (cpf, nome_completo, email, senha, data_nascimento, numero_endereco, cep, bairro, estado, cidade) 
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
        """
        valores = (
            cpf_usr.value, nome_usr.value, email_usr.value, senha_usr.value, 
            nascimento_usr.value, num_usr.value, cep_usr.value, bairro_usr.value, 
            estado_usr.value, cidade_usr.value
        )
        cursor.execute(query, valores)
        con.commit()
        cursor.close()
        pn.state.notifications.success("Usuário inserido com sucesso!")
        return queryAllUsuarios()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Erro ao inserir: {str(e)}', alert_type='danger')

def on_atualizar_usuario():
    try:
        cursor = con.cursor()
        query = """
            UPDATE usuario 
            SET cpf = %s, nome_completo = %s, email = %s, data_nascimento = %s, 
                numero_endereco = %s, cep = %s, bairro = %s, estado = %s, cidade = %s
            WHERE id_usuario = %s;
        """
        # Nota: A senha não é atualizada aqui para simplificar a segurança
        valores = (
            cpf_usr.value, nome_usr.value, email_usr.value, nascimento_usr.value, 
            num_usr.value, cep_usr.value, bairro_usr.value, estado_usr.value, 
            cidade_usr.value, id_usr.value
        )
        cursor.execute(query, valores)
        con.commit()
        cursor.close()
        pn.state.notifications.success("Dados atualizados com sucesso!")
        return queryAllUsuarios()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Erro ao atualizar: {str(e)}', alert_type='danger')

def on_excluir_usuario():
    try:
        cursor = con.cursor()
        cursor.execute("DELETE FROM usuario WHERE id_usuario = %s;", (id_usr.value,))
        con.commit()
        cursor.close()
        pn.state.notifications.success("Usuário removido!")
        return queryAllUsuarios()
    except Exception as e:
        con.rollback()
        # Tratamento especial de Foreign Key alertando sobre a necessidade de Soft Delete
        if 'violates foreign key constraint' in str(e).lower():
            erro_msg = "ERRO: Não pode excluir este usuário pois ele já possui tarefas, colheitas ou ferramentas associadas! (Implemente o Soft Delete)."
        else:
            erro_msg = f'Erro ao excluir: {str(e)}'
        return pn.pane.Alert(erro_msg, alert_type='danger')

# ---------------------------------------------------------
# AJUSTE PANEL: Garantindo que as funções só rodem no CLIQUE
# ---------------------------------------------------------
last_clicks = {'ins': 0, 'atu': 0, 'exc': 0}

def usuario_table_creator(cons, ins, atu, exc):
    global last_clicks
    if ins > last_clicks['ins']:
        last_clicks['ins'] = ins
        return on_inserir_usuario()
    if atu > last_clicks['atu']:
        last_clicks['atu'] = atu
        return on_atualizar_usuario()
    if exc > last_clicks['exc']:
        last_clicks['exc'] = exc
        return on_excluir_usuario()
    # Se for só consultar, retorna a tabela atualizada
    return queryAllUsuarios()

interactive_table_usr = pn.bind(
    usuario_table_creator, 
    btn_consultar_usr.param.clicks, 
    btn_inserir_usr.param.clicks, 
    btn_atualizar_usr.param.clicks, 
    btn_excluir_usr.param.clicks
)

# Montagem do Layout: Formulário de 2 colunas à esquerda e a tabela à direita
modulo_acessos = pn.Row(
    pn.Column(
        '### 👤 Gestão de Usuários',
        id_usr,
        pn.Row(cpf_usr, nascimento_usr),
        nome_usr,
        email_usr,
        senha_usr,
        '#### 📍 Endereço',
        pn.Row(cep_usr, num_usr),
        pn.Row(cidade_usr, estado_usr),
        bairro_usr,
        pn.Row(btn_inserir_usr, btn_consultar_usr),
        pn.Row(btn_atualizar_usr, btn_excluir_usr),
        width=400
    ),
    pn.Column(
        "## 📋 Listagem de Participantes (Core)",
        interactive_table_usr,
        margin=(10, 0, 0, 20)
    )
)

modulo_acessos

In [ ]:
# ==============================================================================
# MÓDULO DE OPERAÇÕES
# ==============================================================================

# Widgets da Tarefa
id_tarefa = pn.widgets.IntInput(name="ID da Tarefa (Alterar/Excluir)", value=0, start=0)
tipo_tarefa = pn.widgets.TextInput(name="📋 Tipo/Descrição da Atividade", value='', placeholder='Ex: Adubação Orgânica, Rega Geral, Poda')
status_tarefa = pn.widgets.Select(name="⚙️ Status Atual", options=['Pendente', 'Em Andamento', 'Concluído'], value='Pendente')
id_usuario_responsavel = pn.widgets.IntInput(name="👤 ID do Usuário Responsável", value=1, start=1)
id_canteiro_alvo = pn.widgets.IntInput(name="🥬 ID do Canteiro Alvo", value=1, start=1)

# Widgets de Consumo de Insumos (Tabela Associativa 'uso')
id_insumo_gasto = pn.widgets.IntInput(name="📦 ID do Insumo Utilizado (Opcional, 0 se nenhum)", value=0, start=0)
qtd_insumo_gasta = pn.widgets.FloatInput(name="🔢 Quantidade Consumida", value=0.0, start=0.0)

# Botões de Ação
btn_consultar_op = pn.widgets.Button(name='Consultar Cronograma', button_type='default')
btn_inserir_op = pn.widgets.Button(name='Agendar Atividade + Insumo', button_type='default')
btn_atualizar_op = pn.widgets.Button(name='Atualizar Status', button_type='default')
btn_excluir_op = pn.widgets.Button(name='Cancelar Atividade', button_type='default')

def queryAllOperacoes():
    """Realiza a junção (JOIN) entre Tarefas, Usuários e o consumo na tabela associativa 'uso'."""
    query = """
        SELECT 
            t.id_tarefa,
            t.tipo AS atividade,
            t.status,
            u.nome_completo AS responsavel,
            cant.id_canteiro AS canteiro,
            i.nome AS insumo_usado,
            uso.quantidade AS qtd_gasta
        FROM tarefa t
        JOIN usuario u ON t.id_usuario = u.id_usuario
        JOIN canteiro cant ON t.id_canteiro = cant.id_canteiro
        LEFT JOIN uso ON t.id_tarefa = uso.id_tarefa
        LEFT JOIN insumos i ON uso.id_insumos = i.id_insumos
        ORDER BY t.id_tarefa DESC;
    """
    df = pd.read_sql_query(query, cnx)
    
    # Trata valores nulos para tarefas que não consumiram insumos
    if not df.empty:
        df['insumo_usado'] = df['insumo_usado'].fillna('Nenhum')
        df['qtd_gasta'] = df['qtd_gasta'].fillna(0.0)
        
    return pn.widgets.Tabulator(df, layout='fit_columns', pagination='remote', page_size=10)

def on_inserir_operacao():
    """Insere a tarefa e, caso um insumo tenha sido associado, alimenta a entidade 'uso'."""
    try:
        cursor = con.cursor()
        
        # 1. Cria a tarefa e recupera o ID serial gerado pelo banco
        query_tarefa = """
            INSERT INTO tarefa (tipo, status, id_usuario, id_canteiro) 
            VALUES (%s, %s, %s, %s) RETURNING id_tarefa;
        """
        cursor.execute(query_tarefa, (tipo_tarefa.value_input, status_tarefa.value, id_usuario_responsavel.value, id_canteiro_alvo.value))
        nova_id_tarefa = cursor.fetchone()[0]
        
        # 2. Se o ID do insumo for válido (> 0) e houver quantidade, registra na tabela associativa 'uso'
        if id_insumo_gasto.value > 0 and qtd_insumo_gasta.value > 0:
            query_uso = "INSERT INTO uso (id_tarefa, id_insumos, quantidade) VALUES (%s, %s, %s);"
            cursor.execute(query_uso, (nova_id_tarefa, id_insumo_gasto.value, qtd_insumo_gasta.value))
            
        con.commit()
        cursor.close()
        pn.state.notifications.success("Atividade operacional registrada!")
        return queryAllOperacoes()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Erro ao agendar atividade: {str(e)}', alert_type='danger')

def on_atualizar_operacao():
    """Atualiza o status ou a descrição de uma atividade em andamento."""
    try:
        cursor = con.cursor()
        query = "UPDATE tarefa SET tipo = %s, status = %s WHERE id_tarefa = %s;"
        cursor.execute(query, (tipo_tarefa.value_input, status_tarefa.value, id_tarefa.value))
        con.commit()
        cursor.close()
pn.state.notifications.success("Status da atividade atualizado!")
        return queryAllOperacoes()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Erro ao atualizar: {str(e)}', alert_type='danger')

def on_excluir_operacao():
    """Remove com segurança os vínculos da tabela de consumo antes de excluir a tarefa mãe."""
    try:
        cursor = con.cursor()
        # 1. Limpa os insumos consumidos por essa tarefa (tabela associativa 'uso')
        cursor.execute("DELETE FROM uso WHERE id_tarefa = %s;", (id_tarefa.value,))
        # 2. Exclui a tarefa fisicamente
        cursor.execute("DELETE FROM tarefa WHERE id_tarefa = %s;", (id_tarefa.value,))
        con.commit()
        cursor.close()
        pn.state.notifications.success("Atividade cancelada e removida do cronograma!")
        return queryAllOperacoes()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Erro ao excluir atividade: {str(e)}', alert_type='danger')

def operacao_table_creator(cons, ins, atu, exc):
    if ins: return on_inserir_operacao()
    if atu: return on_atualizar_operacao()
    if exc: return on_excluir_operacao()
    return queryAllOperacoes()

interactive_table_op = pn.bind(
    operacao_table_creator, 
    btn_consultar_op, btn_inserir_op, btn_atualizar_op, btn_excluir_op
)

# Montagem estrutural do Layout da Tela
modulo_operacoes = pn.Row(
    pn.Column(
        '### 🛠️ Gerenciamento de Atividades',
        id_tarefa,
        tipo_tarefa,
        status_tarefa,
        pn.Row(id_usuario_responsavel, id_canteiro_alvo),
        '### 🧺 Consumo de Materiais (M-N)',
        pn.Row(id_insumo_gasto, qtd_insumo_gasta),
        pn.Row(btn_consultar_op, btn_inserir_op),
        pn.Row(btn_atualizar_op, btn_excluir_op),
        width=380
    ),
    pn.Column(
        "## 📅 Cronograma de Manutenção e Auditoria de Insumos",
        interactive_table_op,
        margin=(10, 0, 0, 20)
    )
)

modulo_operacoes